# Datamine SURPOI Process Example

This notebook demonstrates how to configure and run the **`surpoi`** process wrapper in `dmstudio`.

## Process Description

## SURPOI

Interpolate upper and/or lower elevation values onto a file of two-dimensionally defined points.

The **SURPOI** process can be used to interpolate a variable onto a set of point data. The common application of this process for underground mine planning is to interpolate a set of Z elevations from survey points onto a digitized set of strings and perimeters, which have the correct X and Y coordinates but not the correct elevations.

The output will be completely three dimensional strings or perimeter that can be subsequently used in **DRIVES** to generate wireframe models of existing excavations.

### Input Files:

* **in** (Undefined):
  The input data file, which must contain at least two fields which define the two-
  dimensional position of a series of points.
  Required=Yes

* **elev** (Undefined):
  Input file containing elevation data that will be used to control the interpolation. It
  must contain the fields **X** , **Y** and **UPPER** , and may optionally contain the
  field **LOWER**.
  Required=Yes

* **trend** (Undefined):
  Optional input trend file.
  Required=No

### Output Files:

* **out** (Undefined):
  Output file that will contain all the fields that were in the input file, plus the
  interpolated elevation fields.
  Required=Yes

### Fields:

* **xin** (Numeric : IN):
  Name of the X field in the input data file.
  Default=Undefined
  Required=Yes

* **yin** (Numeric : IN):
  Name of the Y field in the input data file.
  Default=Undefined
  Required=Yes

### Parameters:

* **radius**:
  Search radius for interpolation [the default is the mean spacing between the elevation
  data points].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **power**:
  Power to be used for inverse power of distance weighting (2).
  Range=Undefined
  Values=Undefined
  Default=2
  Required=No

* **minnop**:
  Minimum number of samples required for interpolation (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surpoi')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surpoi
print("Running surpoi...")
dm_cmd.surpoi(
    in_i='t_assays',  # required
    elev_i='t_input_file',  # required
    trend_i='optional',  # required
    out_o='t_surpoi_out',  # required
    xin_f='optional',  # required
    yin_f='optional',  # required
    # radius_p='optional',  # optional
    # power_p=2,  # optional
    # minnop_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surpoi execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_surpoi_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")